# hudukaata — Search (Google Colab)

Browse and search the media index built by `indexer.ipynb`.

**Run all cells in order, top to bottom.** Re-running individual cells out of
order may cause import errors.

**Steps:**
1. Configure your Drive paths (Cell 1)
2. Mount Google Drive (Cell 2)
3. Install packages (Cell 3)
4. Load the index (Cell 4)
5. Browse all face clusters (Cell 5) — re-run freely
6. Search by text and optional face filter (Cell 6) — re-run freely

## 1 — Configuration

| Variable | Description |
|---|---|
| `STORE_DRIVE_PATH` | Subfolder in MyDrive where the index was written by the indexer notebook. |
| `MEDIA_DRIVE_PATH` | Subfolder in MyDrive where your original media files live. |
| `INDEXER_KEY` | Must match the model used when indexing: `"blip2-sentok-exif-insightface"` for `blip2_faces`; `"blip2-sentok-exif"` for `blip2`. |
| `TOP_K` | Number of search results to return per query. |
| `FACE_DISPLAY_LIMIT` | Maximum number of face clusters to show in Cell 5. |

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
STORE_DRIVE_PATH  = "hudukaata-store"               # must match what the indexer wrote
MEDIA_DRIVE_PATH  = "Photos"                        # where original media files live
INDEXER_KEY       = "blip2-sentok-exif-insightface" # or "blip2-sentok-exif"
TOP_K             = 10
FACE_DISPLAY_LIMIT = 200                            # max face clusters shown in Cell 5
# ──────────────────────────────────────────────────────────────────────────────

## 2 — Mount Google Drive

A browser pop-up will ask you to authorise access.

In [ ]:
from google.colab import drive as _colab_drive
_colab_drive.mount("/content/drive", force_remount=False)
print("Drive mounted at /content/drive/MyDrive/")

## 3 — Install packages

In [ ]:
BRANCH = "main"
# BRANCH = "claude/add-colab-run-feature-KFPa5"  # use this branch before it is merged

import subprocess, sys

def _pip(*args):
    # -q suppresses progress bars but keeps warnings visible.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

_pip(f"git+https://github.com/kusimari/hudukaata@{BRANCH}#subdirectory=common")
# indexer is installed for its ChromaDB store classes used by the search loader.
_pip(f"git+https://github.com/kusimari/hudukaata@{BRANCH}#subdirectory=indexer")
_pip(f"git+https://github.com/kusimari/hudukaata@{BRANCH}#subdirectory=search")

## 4 — Load the index

Loads the vector index from Google Drive directly into memory — no server process needed.

In [ ]:
import logging, os
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s %(message)s")

os.environ["SEARCH_STORE"]       = f"file:///content/drive/MyDrive/{STORE_DRIVE_PATH}"
os.environ["SEARCH_MEDIA"]       = f"gdrive:///{MEDIA_DRIVE_PATH}"
os.environ["SEARCH_INDEXER_KEY"] = INDEXER_KEY
os.environ["SEARCH_TOP_K"]       = str(TOP_K)

from search.config import Settings
from search.startup import load

ctx = load(Settings())
print("Index loaded.")
if ctx.face_store is not None:
    print("Face store available — face browsing and filtering enabled.")
else:
    print("No face store (re-index with CAPTION_MODEL='blip2_faces' to enable face features).")

## 5 — Browse all face clusters

Displays a representative thumbnail for each face cluster, sorted by frequency
(most-seen faces first). Note the **Cluster ID** shown under each image — pass
it to `FACE_IDS` in Cell 6 to restrict search results to photos containing that person.

In [ ]:
import html as _html
from IPython.display import display as _display, Image as _Image, HTML as _HTML
from pathlib import Path

if ctx.face_store is None:
    print("Face store not available. Re-index with CAPTION_MODEL='blip2_faces'.")
else:
    faces = ctx.face_store.list_all(FACE_DISPLAY_LIMIT)
    if not faces:
        print("No face clusters found in the index yet.")
    else:
        print(f"{len(faces)} face cluster(s) shown (limit: {FACE_DISPLAY_LIMIT}).\n")
        for face in faces:
            img_path = Path("/content/drive/MyDrive") / MEDIA_DRIVE_PATH / face.relative_path
            count    = int(face.extra.get("count", 0))
            safe_id  = _html.escape(str(face.id))
            _display(_HTML(
                f"<b>Cluster ID:</b> <code>{safe_id}</code> &nbsp; "
                f"<b>Photos:</b> {count}"
            ))
            if img_path.is_file():
                _display(_Image(filename=str(img_path), width=200))
            else:
                _display(_HTML("<i>(thumbnail not found on Drive)</i>"))

## 6 — Search by text (and optionally by face)

Set `QUERY` to any natural-language description.
Set `FACE_IDS` to a list of Cluster IDs from Cell 5 to restrict results to photos
containing those people (e.g. `["abc123", "def456"]`). Leave it empty to search
across all photos.

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
QUERY    = "sunset at the beach"
FACE_IDS = []  # copy Cluster IDs from Cell 5, e.g. ["abc123", "def456"]
# ──────────────────────────────────────────────────────────────────────────────

import html as _html
from IPython.display import display as _display, Image as _Image, HTML as _HTML
from pathlib import Path
from common.index import CaptionItem

if not QUERY.strip():
    print("QUERY is empty — please set it above.")
else:
    results = ctx.index_store.search(CaptionItem(text=QUERY), TOP_K)

    if FACE_IDS and ctx.face_store is not None:
        # Build a lookup of cluster_id → image_paths from a single list_all call.
        all_faces = ctx.face_store.list_all(10_000)
        face_index = {item.id: item for item in all_faces}
        face_filter: set[str] = set()
        for fid in FACE_IDS:
            cluster = face_index.get(fid)
            if cluster is None:
                print(f"Warning: face cluster {fid!r} not found in the index.")
                continue
            paths = cluster.extra.get("image_paths", "")
            face_filter.update(p for p in paths.split(",") if p)
        results = [r for r in results if r.relative_path in face_filter]

    if not results:
        print("No results found.")
    else:
        print(f"{len(results)} result(s) for: {QUERY!r}\n")
        for r in results:
            img_path   = Path("/content/drive/MyDrive") / MEDIA_DRIVE_PATH / r.relative_path
            safe_path  = _html.escape(str(r.relative_path))
            _display(_HTML(f"<b>{safe_path}</b> &nbsp; score: {r.score:.3f}"))
            if img_path.is_file():
                _display(_Image(filename=str(img_path), width=400))
            else:
                _display(_HTML("<i>(file not found on Drive)</i>"))